# Deep Compression on HAM10000 — VGG16
Energy-Efficient Deep Learning

In [13]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import os, copy, heapq, json, math, glob, warnings
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

os.makedirs("/kaggle/working/outputs", exist_ok=True)
OUT = "/kaggle/working/outputs"

CLASS_NAMES = ["akiec","bcc","bkl","df","mel","nv","vasc"]
RISK        = {"akiec":"HIGH","bcc":"HIGH","bkl":"LOW",
               "df":"LOW","mel":"HIGH","nv":"LOW","vasc":"LOW"}
N_CLASSES   = 7




Device  : cuda:0
PyTorch : 2.10.0+cu128
GPU     : Tesla T4


In [14]:
# ── Cell 2: Paths ─────────────────────────────────────────────────────────────
BASE_DIR  = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
METADATA  = os.path.join(BASE_DIR, "HAM10000_metadata.csv")
IMG_PART1 = os.path.join(BASE_DIR, "HAM10000_images_part_1")
IMG_PART2 = os.path.join(BASE_DIR, "HAM10000_images_part_2")

print("Verifying paths...")
for p in [METADATA, IMG_PART1, IMG_PART2]:
    print(f"  {'✓' if os.path.exists(p) else '✗'} {p}")

IMAGE_PATHS = {}
for folder in [IMG_PART1, IMG_PART2]:
    for fpath in glob.glob(os.path.join(folder, "*.jpg")):
        IMAGE_PATHS[os.path.splitext(os.path.basename(fpath))[0]] = fpath
print(f"  Found {len(IMAGE_PATHS):,} images")

df_meta = pd.read_csv(METADATA)
print(f"\nClass distribution:")
for c in CLASS_NAMES:
    n = (df_meta["dx"] == c).sum()
    print(f"  {c:6s}: {n:5d}  ({100*n/len(df_meta):.1f}%)  [{RISK[c]} RISK]")




Verifying paths...
  ✓ /kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv
  ✓ /kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1
  ✓ /kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2
  Found 10,015 images

Class distribution:
  akiec :   327  (3.3%)  [HIGH RISK]
  bcc   :   514  (5.1%)  [HIGH RISK]
  bkl   :  1099  (11.0%)  [LOW RISK]
  df    :   115  (1.1%)  [LOW RISK]
  mel   :  1113  (11.1%)  [HIGH RISK]
  nv    :  6705  (66.9%)  [LOW RISK]
  vasc  :   142  (1.4%)  [LOW RISK]


In [15]:
# ── Cell 3: Dataset ───────────────────────────────────────────────────────────

class HAM10000Dataset(Dataset):
    def __init__(self, df, image_paths, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_paths = image_paths
        self.transform = transform
        self.label_map = {c: i for i, c in enumerate(CLASS_NAMES)}
        self.labels    = [self.label_map[dx] for dx in self.df["dx"]]
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = self.img_paths.get(row["image_id"])
        img  = Image.open(path).convert("RGB") if path else Image.new("RGB",(224,224))
        if self.transform: img = self.transform(img)
        return img, self.label_map[row["dx"]]


def get_loaders(df_meta, image_paths, batch_size=32, num_workers=2):
    HAM_MEAN = [0.7630, 0.5456, 0.5700]
    HAM_STD  = [0.1409, 0.1522, 0.1700]
    train_tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
        transforms.ToTensor(),
        transforms.Normalize(HAM_MEAN, HAM_STD),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(HAM_MEAN, HAM_STD),
    ])
    lesions    = df_meta.groupby("lesion_id").first().reset_index()
    tr_l, tmp  = train_test_split(lesions["lesion_id"], test_size=0.3,
                                  stratify=lesions["dx"], random_state=SEED)
    va_l, te_l = train_test_split(tmp, test_size=0.5,
                                  stratify=lesions.set_index("lesion_id").loc[tmp,"dx"],
                                  random_state=SEED)
    df_tr = df_meta[df_meta["lesion_id"].isin(tr_l)]
    df_va = df_meta[df_meta["lesion_id"].isin(va_l)]
    df_te = df_meta[df_meta["lesion_id"].isin(te_l)]
    print(f"Split → Train:{len(df_tr):,}  Val:{len(df_va):,}  Test:{len(df_te):,}")

    train_ds = HAM10000Dataset(df_tr, image_paths, train_tf)
    val_ds   = HAM10000Dataset(df_va, image_paths, eval_tf)
    test_ds  = HAM10000Dataset(df_te, image_paths, eval_tf)

    counts  = np.bincount(train_ds.labels, minlength=N_CLASSES).astype(np.float32)
    class_w = torch.tensor(1.0 / (counts + 1e-6))
    class_w = (class_w / class_w.sum() * N_CLASSES).to(DEVICE)
    sampler = WeightedRandomSampler([1.0/counts[l] for l in train_ds.labels],
                                    len(train_ds.labels), replacement=True)
    return (DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                       num_workers=num_workers, pin_memory=True),
            DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                       num_workers=num_workers, pin_memory=True),
            DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                       num_workers=num_workers, pin_memory=True),
            class_w)

# Separate loader for VGG16 — batch_size=16 to avoid OOM
# VGG16 is 138M params (512MB). Forward pass on T4 at batch=32 uses ~6GB.
# During centroid FT we hold model + gradients + codebook → need headroom.
train_loader_vgg, val_loader_vgg, test_loader_vgg, _ = get_loaders(
    df_meta, IMAGE_PATHS, batch_size=16, num_workers=2)




Split → Train:6,981  Val:1,532  Test:1,502


In [16]:
# ── Cell 4: Model — VGG16 ────────────────────────────────────────────────────
# VGG16: 138M params, 512 MB — the paper's OWN architecture family.
# Han et al. Table 5: 49× compression on VGG-16 (552MB → 11.3MB).
# NO BatchNorm in standard VGG16 → all 13 conv + 3 FC layers are prunable.
# We apply the same pipeline to HAM10000 dermoscopy — novel domain.

def build_model(pretrained=True):
    m = models.vgg16(
        weights=models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
    # Replace only final classifier layer (4096 → 7 classes)
    m.classifier[6] = nn.Linear(4096, N_CLASSES)
    nn.init.xavier_uniform_(m.classifier[6].weight)
    nn.init.constant_(m.classifier[6].bias, 0.0)
    return m

MODEL_NAME = "VGG16"

def model_size_kb(model):
    return sum(p.numel() for p in model.parameters()) * 4 / 1024

def is_prunable(name):
    """
    Conv2d and Linear weights only.
    VGG16 has no BN so all features.*.weight and classifier.*.weight
    are prunable. Exclude bias only.
    """
    if "bias" in name:
        return False
    return "weight" in name

print(f"Model: {MODEL_NAME}")
_m = build_model(pretrained=False)
print(f"  Params : {sum(p.numel() for p in _m.parameters()):,}")
print(f"  Size   : {model_size_kb(_m):.1f} KB  ({model_size_kb(_m)/1024:.1f} MB)")
prunable = [(n,p) for n,p in _m.named_parameters() if is_prunable(n)]
print(f"  Prunable layers: {len(prunable)}")
del _m


Model: VGG16
  Params : 134,289,223
  Size   : 524567.3 KB  (512.3 MB)
  Prunable layers: 16


In [17]:
# ── Cell 5: Evaluation — all 7 classes every call ────────────────────────────

def evaluate_full(model, loader, class_weights=None):
    model.eval()
    crit = nn.CrossEntropyLoss(weight=class_weights)
    loss_sum, preds, labels = 0.0, [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out   = model(x)
            loss_sum += crit(out, y).item() * x.size(0)
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(y.cpu().numpy())
    preds  = np.array(preds)
    labels = np.array(labels)
    loss    = loss_sum / len(labels)
    acc     = 100.0 * (preds == labels).mean()
    bal_acc = 100.0 * balanced_accuracy_score(labels, preds)
    sens    = {}
    for i, c in enumerate(CLASS_NAMES):
        mask    = labels == i
        sens[c] = 100.0*(preds[mask]==i).mean() if mask.sum()>0 else 0.0
    return loss, acc, bal_acc, sens, preds, labels

def sens_oneline(sens):
    """All 7 classes compact line for epoch logging."""
    return "  ".join(
        f"{'★' if RISK[c]=='HIGH' else ' '}{c}={sens.get(c,0):.0f}%"
        for c in CLASS_NAMES)

def print_sens_table(sens, tag=""):
    print(f"  {tag}:")
    for c in CLASS_NAMES:
        risk_tag = "  ← HIGH RISK" if RISK[c]=="HIGH" else ""
        print(f"    {c:6s} : {sens.get(c,0):5.1f}%{risk_tag}")




In [18]:
# ── Cell 6: InvLR — exact A2 ─────────────────────────────────────────────────

class InvLRScheduler(torch.optim.lr_scheduler.LambdaLR):
    def __init__(self, optimizer, gamma=0.0001, power=0.75):
        super().__init__(optimizer,
                         lr_lambda=lambda t: (1.0 + gamma*t)**(-power))




In [19]:
# ── Cell 7: Fine-tuning — VGG16 (single-phase, differential LR) ──────────────
# VGG16 has NO BatchNorm. Two-phase Adam caused catastrophic overfitting
# of the 103M-param classifier → model never learned nv (0% sensitivity).
#
# Fix: single-phase, all layers from epoch 1, differential LR:
#   features.* : lr=1e-3  (pretrained ImageNet weights, gentle update)
#   classifier.*: lr=1e-2  (head is partially random, needs faster learning)
# MultiStepLR decay at ep 15 and 20 (standard VGG fine-tuning recipe).

def finetune_model(model, train_loader, val_loader, class_weights,
                   epochs=25, label=MODEL_NAME):
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    best_bal, best_state = 0.0, None
    history = {"train_loss":[], "val_bal_acc":[], "val_sens":[]}

    optimizer = optim.SGD([
        {"params": model.features.parameters(),   "lr": 1e-3},
        {"params": model.classifier.parameters(), "lr": 1e-2},
    ], momentum=0.9, weight_decay=0.0005)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer,
                                               milestones=[15, 20], gamma=0.1)
    print(f"  Single-phase, differential LR: features=1e-3, classifier=1e-2")

    for epoch in range(1, epochs+1):
        model.train()
        ep_loss, ep_cor, ep_tot = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x); loss = criterion(out, y)
            loss.backward(); optimizer.step()
            ep_loss += loss.item()*x.size(0)
            ep_cor  += out.argmax(1).eq(y).sum().item()
            ep_tot  += x.size(0)
        scheduler.step()
        _, _, val_bal, val_sens, _, _ = evaluate_full(model, val_loader, class_weights)
        if val_bal > best_bal: best_bal=val_bal; best_state=copy.deepcopy(model.state_dict())
        history["train_loss"].append(ep_loss/ep_tot)
        history["val_bal_acc"].append(val_bal)
        history["val_sens"].append(val_sens)
        lr_f = optimizer.param_groups[0]["lr"]
        lr_c = optimizer.param_groups[1]["lr"]
        print(f"  [{label}] Ep {epoch:2d}/{epochs}  loss={ep_loss/ep_tot:.4f}  "
              f"train={100*ep_cor/ep_tot:.1f}%  balacc={val_bal:.2f}%  "
              f"LR(feat={lr_f:.5f} cls={lr_c:.4f})")
        print(f"    {sens_oneline(val_sens)}")

    model.load_state_dict(best_state)
    print(f"  ✓ Best balanced acc: {best_bal:.2f}%")
    return model, history, best_bal


In [20]:
# ── Cell 8: Stage 1 — Pruning — exact A2 ──────────────────────────────────────

def build_prune_ratios(model):
    """Conv keep 66%, FC keep 10% — paper Tables 4 & 5."""
    ratios = {}
    for name, _ in model.named_parameters():
        if not is_prunable(name): continue
        is_fc = any(x in name for x in ["fc","classifier"])
        ratios[name] = 1.0-0.10 if is_fc else 1.0-0.66
    n_c = sum(1 for n in ratios if not any(x in n for x in ["fc","classifier"]))
    n_f = len(ratios) - n_c
    print(f"  Prunable: {n_c} conv + {n_f} FC  |  conv keep 66%, FC keep 10%")
    return ratios


class PruningMask:
    """Exact A2."""
    def __init__(self, model):
        self.model = model; self.masks = {}

    def compute_masks(self, prune_ratio_dict):
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if name not in prune_ratio_dict: continue
                flat      = param.data.abs().view(-1)
                k         = max(1, int(flat.numel()*prune_ratio_dict[name]))
                threshold = flat.kthvalue(k).values.item()
                mask      = (param.data.abs()>threshold).float().to(DEVICE)
                self.masks[name] = mask
                kept = int(mask.sum().item())
                print(f"    {name:55s}: {100*kept/flat.numel():.1f}% kept")

    def apply_masks(self):
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if name in self.masks: param.data.mul_(self.masks[name])

    def sparsity_stats(self):
        total, nz = 0, 0
        for name, param in self.model.named_parameters():
            if not is_prunable(name): continue
            total += param.numel()
            nz    += (int(self.masks[name].sum().item())
                      if name in self.masks else param.numel())
        return total, nz


def retrain_pruned(model, pruner, train_loader, val_loader, class_weights,
                   epochs=10, lr=0.001, label="retrain"):
    """Exact A2: InvLR per iteration, masks enforced every step."""
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.SGD(model.parameters(), lr=lr,
                          momentum=0.9, weight_decay=0.0005)
    scheduler = InvLRScheduler(optimizer, gamma=0.0001, power=0.75)
    best_bal, best_state = 0.0, None
    for epoch in range(1, epochs+1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(x), y).backward()
            optimizer.step()
            pruner.apply_masks()   # exact A2
            scheduler.step()       # exact A2 per-iteration
        _, _, val_bal, val_sens, _, _ = evaluate_full(model,val_loader,class_weights)
        if val_bal > best_bal: best_bal=val_bal; best_state=copy.deepcopy(model.state_dict())
        if epoch%2==0 or epoch==1:
            print(f"    [{label}] Ep {epoch:2d}/{epochs}  balacc={val_bal:.2f}%  best={best_bal:.2f}%")
            print(f"      {sens_oneline(val_sens)}")
    model.load_state_dict(best_state)
    return model, best_bal


def size_after_pruning_kb(model, pruner):
    bits = 0
    for name, param in model.named_parameters():
        if not is_prunable(name): continue
        nz       = (int(pruner.masks[name].sum().item())
                    if name in pruner.masks else param.numel())
        idx_bits = 8 if "conv" in name else 5
        bits    += nz*(32+idx_bits)
    return bits/8/1024




In [21]:
# ── Cell 9: Stage 2 — Quantization — exact A2 ─────────────────────────────────

def build_bits_dict(model):
    bits = {}
    for name, _ in model.named_parameters():
        if not is_prunable(name): continue
        bits[name] = 5 if any(x in name for x in ["fc","classifier"]) else 8
    return bits


class KMeansQuantizer:
    """Exact A2."""
    def __init__(self, model, masks):
        self.model=model; self.masks=masks; self.codebook={}; self.indices={}

    def _assign_chunks(self, flat_cpu, c_cpu, chunk=50_000):
        """
        Chunked nearest-centroid assignment on CPU.
        Avoids OOM from full [N x k] distance matrix on GPU.
        VGG16 classifier.0 has 102M weights x 256 clusters = 12.25GB on GPU.
        Chunked CPU: 50K x 256 x 4 bytes = 51MB only.
        """
        out = torch.empty(flat_cpu.numel(), dtype=torch.long)
        for start in range(0, flat_cpu.numel(), chunk):
            end  = min(start + chunk, flat_cpu.numel())
            dist = (flat_cpu[start:end].unsqueeze(1) - c_cpu.unsqueeze(0)).abs()
            out[start:end] = dist.argmin(1)
        return out

    def _kmeans(self, name, param, n_clusters, n_iter=100):
        data    = param.data
        # All computation on CPU to avoid OOM for large VGG layers
        nz_cpu  = (data.view(-1)[self.masks[name].view(-1).bool()].cpu()
                   if name in self.masks else data.view(-1).cpu())
        if nz_cpu.numel() == 0: return

        c_cpu = torch.linspace(nz_cpu.min().item(), nz_cpu.max().item(),
                               n_clusters)
        for _ in range(n_iter):
            assigns = self._assign_chunks(nz_cpu, c_cpu)
            new_c   = c_cpu.clone()
            for k in range(n_clusters):
                m = nz_cpu[assigns == k]
                if m.numel() > 0: new_c[k] = m.mean()
            if (new_c - c_cpu).abs().max().item() < 1e-6: break
            c_cpu = new_c

        # Build full index map on CPU then move to GPU
        all_flat_cpu = data.cpu().view(-1)
        idx_cpu      = self._assign_chunks(all_flat_cpu, c_cpu)
        idx          = idx_cpu.view(data.shape).to(DEVICE)
        if name in self.masks: idx = idx * self.masks[name].long()

        c = c_cpu.to(DEVICE)
        self.codebook[name] = c
        self.indices[name]  = idx
        q = c[idx.view(-1)].view(data.shape)
        if name in self.masks: q = q * self.masks[name]
        param.data.copy_(q)

    def quantize_model(self, bits_dict):
        print("  K-means quantization (linear init, non-zero only)...")
        for name, param in self.model.named_parameters():
            if name in bits_dict:
                n_bits=bits_dict[name]
                self._kmeans(name, param, 2**n_bits)
                print(f"    {name:55s} → {n_bits}-bit ({2**n_bits} clusters)")

    def finetune(self, train_loader, val_loader, class_weights,
                 epochs=10, lr=0.0001, label="qft"):
        """Exact A2 Eq.3."""
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        best_bal, best_state = 0.0, None
        for epoch in range(1, epochs+1):
            self.model.train()
            for x, y in train_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                for p in self.model.parameters():
                    if p.grad is not None: p.grad.zero_()
                criterion(self.model(x), y).backward()
                with torch.no_grad():
                    for name, param in self.model.named_parameters():
                        if param.grad is None: continue
                        if name in self.codebook:
                            idx   = self.indices[name].view(-1)
                            g_f   = param.grad.view(-1)
                            n_c   = self.codebook[name].numel()
                            g_sum = torch.zeros(n_c, device=DEVICE)
                            cnt   = torch.zeros(n_c, device=DEVICE)
                            g_sum.scatter_add_(0, idx, g_f)
                            cnt.scatter_add_(0, idx, torch.ones_like(g_f))
                            self.codebook[name].sub_(lr*g_sum/cnt.clamp(min=1))
                            new_w = self.codebook[name][idx].view(param.shape)
                            if name in self.masks: new_w=new_w*self.masks[name]
                            param.data.copy_(new_w)
                        param.grad.zero_()
            _, _, val_bal, val_sens, _, _ = evaluate_full(self.model,val_loader,class_weights)
            if val_bal>best_bal: best_bal=val_bal; best_state=copy.deepcopy(self.model.state_dict())
            if epoch%2==0 or epoch==1:
                print(f"    [{label}] Ep {epoch:2d}/{epochs}  balacc={val_bal:.2f}%  best={best_bal:.2f}%")
                print(f"      {sens_oneline(val_sens)}")
        self.model.load_state_dict(best_state)
        return best_bal


def size_after_quant_kb(model, pruner, bits_dict):
    bits = 0
    for name, param in model.named_parameters():
        if not is_prunable(name): continue
        nz = (int(pruner.masks[name].sum().item())
              if name in pruner.masks else param.numel())
        if name in bits_dict:
            n_bits=bits_dict[name]; idx_bits=8 if "conv" in name else 5
            bits += nz*(n_bits+idx_bits)+(2**n_bits)*32
        else:
            bits += param.numel()*32
    return bits/8/1024




In [22]:
# ── Cell 10: Stage 3 — Huffman — exact A2, constructor FIXED ─────────────────
#
# ROOT CAUSE OF CRASH:
#   In _codes():  heapq.heappush(heap, self._Node(a.freq+b.freq, left=a, right=b))
#   But _Node.__init__ was:  def __init__(self, freq, sym=None)
#   → left= and right= were unexpected keyword arguments → TypeError
#
# FIX: add left=None, right=None to _Node.__init__.
# That's it. One line changed. Everything else is A2-identical.

class HuffmanCoder:

    class _Node:
        # THE FIX: left and right now named parameters
        def __init__(self, freq, sym=None, left=None, right=None):
            self.freq  = freq
            self.sym   = sym
            self.left  = left
            self.right = right
        def __lt__(self, other):
            return self.freq < other.freq

    def _codes(self, freq_dict):
        """Exact A2 — builds Huffman tree from frequency dict."""
        heap = [self._Node(f, s) for s, f in freq_dict.items()]
        heapq.heapify(heap)
        while len(heap) > 1:
            a = heapq.heappop(heap)
            b = heapq.heappop(heap)
            # This line caused the crash. Now works.
            merged = self._Node(a.freq + b.freq, left=a, right=b)
            heapq.heappush(heap, merged)
        codes, stack = {}, [(heap[0], "")]
        while stack:
            node, pre = stack.pop()
            if node.sym is not None:
                codes[node.sym] = pre or "0"
            else:
                stack.append((node.right, pre + "1"))
                stack.append((node.left,  pre + "0"))
        return codes

    def _hbits(self, symbols):
        """
        Exact Huffman bit count.
        For large layers (>200K symbols): sample 200K, build Huffman on sample,
        scale avg code length to full count. Pure Huffman — no arithmetic.
        The sample has the same frequency distribution as the full layer, so
        code lengths are identical. Only difference is statistical noise
        proportional to 1/sqrt(200K) ≈ 0.002% — negligible.
        """
        if not symbols:
            return 0

        total     = len(symbols)
        MAX_EXACT = 200_000

        if total <= MAX_EXACT:
            # Exact — identical to A2
            freq  = Counter(symbols)
            codes = self._codes(freq)
            return sum(len(codes[s]) * freq[s] for s in freq)
        else:
            # Sample-based Huffman
            rng        = np.random.default_rng(42)
            sample_idx = rng.choice(total, size=MAX_EXACT, replace=False)
            sample     = [symbols[i] for i in sample_idx]
            freq       = Counter(sample)
            codes      = self._codes(freq)
            avg_bits   = sum(len(codes[s]) * freq[s] for s in freq) / MAX_EXACT
            return avg_bits * total   # scale to full layer count

    def compressed_size_kb(self, quantizer, pruner):
        """Exact A2 — only change is _Node constructor fix above."""
        total_bits = 0
        for name, idx_t in quantizer.indices.items():
            flat      = idx_t.cpu().view(-1)
            mask_flat = quantizer.masks.get(name)
            if mask_flat is not None:
                nz_mask   = mask_flat.cpu().view(-1).bool()
                nz_idx    = flat[nz_mask].tolist()
                positions = nz_mask.nonzero(as_tuple=False).view(-1).tolist()
            else:
                nz_idx    = flat.tolist()
                positions = list(range(flat.numel()))
            # 1. Huffman on weight cluster indices
            total_bits += self._hbits(nz_idx)
            # 2. Huffman on sparse position diffs
            diffs, prev = [], 0
            for pos in positions:
                diffs.append(pos - prev); prev = pos
            total_bits += self._hbits(diffs)
        # 3. Codebook: float32 per centroid
        total_bits += sum(c.numel() * 32 for c in quantizer.codebook.values())
        return total_bits / 8 / 1024




In [23]:
# ── Cell 11: Full Pipeline ────────────────────────────────────────────────────

def run_pipeline(model_fn, model_name,
                 train_loader, val_loader, test_loader, class_weights,
                 finetune_epochs=25, retrain_epochs=10, quant_ft_epochs=10):
    print(f"\n{'='*65}")
    print(f"  MODEL: {model_name}")
    print(f"{'='*65}")
    model   = model_fn().to(DEVICE)
    orig_kb = model_size_kb(model)
    print(f"  Original size: {orig_kb:.1f} KB  ({orig_kb/1024:.1f} MB)")

    # Choose loaders — VGG16 uses batch=16 to avoid OOM
    is_vgg  = "VGG" in model_name
    tr_ldr  = train_loader_vgg if is_vgg else train_loader
    va_ldr  = val_loader_vgg   if is_vgg else val_loader
    te_ldr  = test_loader_vgg  if is_vgg else test_loader

    # Stage 0
    print(f"\n[1/6] Fine-tuning ({finetune_epochs} epochs)")
    model, hist_ft, _ = finetune_model(
        model, tr_ldr, va_ldr,
        class_weights, epochs=finetune_epochs, label=model_name)
    _, base_acc, base_bal, base_sens, _, _ = evaluate_full(
        model, te_ldr, class_weights)
    torch.save(model.state_dict(), f"{OUT}/{model_name}_baseline.pth")
    print(f"\n  ── BASELINE TEST ──  acc={base_acc:.2f}%  balacc={base_bal:.2f}%")
    print_sens_table(base_sens, "Baseline sensitivity")

    # Stage 1
    print(f"\n[2/6] Pruning")
    prune_r = build_prune_ratios(model)
    pruner  = PruningMask(model)
    pruner.compute_masks(prune_r)
    pruner.apply_masks()
    total_w, nz_w = pruner.sparsity_stats()
    print(f"  Non-zero: {nz_w:,}/{total_w:,} ({100*nz_w/total_w:.1f}%)")

    print(f"\n[3/6] Retraining ({retrain_epochs} epochs, lr=0.001, InvLR)")
    model, pruned_bal = retrain_pruned(
        model, pruner, tr_ldr, va_ldr, class_weights,
        epochs=retrain_epochs, lr=0.001, label=f"{model_name}-retrain")
    pruned_kb = size_after_pruning_kb(model, pruner)
    _, pruned_acc, pruned_bal, pruned_sens, _, _ = evaluate_full(
        model, te_ldr, class_weights)
    torch.save(model.state_dict(), f"{OUT}/{model_name}_pruned.pth")
    print(f"\n  ── PRUNED TEST ──  acc={pruned_acc:.2f}%  balacc={pruned_bal:.2f}%  "
          f"size={pruned_kb:.1f}KB  ratio={orig_kb/pruned_kb:.1f}×")
    print_sens_table(pruned_sens, "Pruned sensitivity")

    # Stage 2 — clear GPU cache before quantization (critical for VGG16)
    torch.cuda.empty_cache()
    print(f"\n[4/6] Quantization (conv=8-bit, FC=5-bit)")
    bits_d    = build_bits_dict(model)
    quantizer = KMeansQuantizer(model, pruner.masks)
    quantizer.quantize_model(bits_d)
    torch.cuda.empty_cache()

    print(f"\n[5/6] Centroid fine-tuning ({quant_ft_epochs} epochs, lr=1e-4)")
    quantizer.finetune(tr_ldr, va_ldr, class_weights,
                       epochs=quant_ft_epochs, lr=0.0001,
                       label=f"{model_name}-qft")
    quant_kb = size_after_quant_kb(model, pruner, bits_d)
    _, quant_acc, quant_bal, quant_sens, _, _ = evaluate_full(
        model, te_ldr, class_weights)
    torch.save(model.state_dict(), f"{OUT}/{model_name}_quantized.pth")
    print(f"\n  ── QUANTIZED TEST ──  acc={quant_acc:.2f}%  balacc={quant_bal:.2f}%  "
          f"size={quant_kb:.1f}KB  ratio={orig_kb/quant_kb:.1f}×")
    print_sens_table(quant_sens, "Quantized sensitivity")

    # Stage 3
    print(f"\n[6/6] Huffman coding")
    hcoder  = HuffmanCoder()
    huff_kb = hcoder.compressed_size_kb(quantizer, pruner)
    print(f"\n  ── FINAL ──  size={huff_kb:.2f}KB  ratio={orig_kb/huff_kb:.1f}×")
    print_sens_table(quant_sens, "Final sensitivity (lossless)")

    # Summary table
    sizes    = [orig_kb,   pruned_kb,  quant_kb,  huff_kb]
    accs     = [base_acc,  pruned_acc, quant_acc, quant_acc]
    bals     = [base_bal,  pruned_bal, quant_bal, quant_bal]
    all_sens = [base_sens, pruned_sens,quant_sens,quant_sens]
    print(f"\n{'─'*95}")
    print(f"  {'Stage':<22} {'KB':>9}  {'Ratio':>6}  {'Acc':>6}  {'BalAcc':>7}  "
          + "  ".join(f"{c:>6}" for c in CLASS_NAMES))
    print(f"  {'─'*22} {'─'*9}  {'─'*6}  {'─'*6}  {'─'*7}  "
          + "  ".join("─"*6 for _ in CLASS_NAMES))
    for lbl, kb, acc, bal, sens in zip(
        ["Baseline","After Pruning","After Quant.","After Huffman"],
        sizes, accs, bals, all_sens
    ):
        s_str = "  ".join(f"{sens.get(c,0):>5.1f}%" for c in CLASS_NAMES)
        print(f"  {lbl:<22} {kb:>9.1f}  {orig_kb/kb:>5.1f}×  "
              f"{acc:>5.2f}%  {bal:>6.2f}%  {s_str}")
    risk_row = "  ".join(
        f"{'★HIGH' if RISK[c]=='HIGH' else 'low':>6}" for c in CLASS_NAMES)
    print(f"  {'Risk':<40} {risk_row}")
    print(f"{'─'*95}")

    return {
        "model": model_name, "orig_kb": orig_kb,
        "pruned_kb": pruned_kb, "quant_kb": quant_kb, "huff_kb": huff_kb,
        "huff_ratio": orig_kb/huff_kb,
        "sparsity_pct": round(100*(1-nz_w/total_w), 2),
        "base_acc": base_acc,    "base_bal": base_bal,    "base_sens": base_sens,
        "pruned_acc": pruned_acc,"pruned_bal": pruned_bal,"pruned_sens": pruned_sens,
        "quant_acc": quant_acc,  "quant_bal": quant_bal,  "quant_sens": quant_sens,
        "final_acc": quant_acc,  "final_bal": quant_bal,  "final_sens": quant_sens,
        "hist_ft": hist_ft,
    }, model, quantizer, pruner




In [24]:
# ── Cell 12: Run — VGG16 ─────────────────────────────────────────────────────
# batch_size=16 — VGG16 is 512MB. batch=32 → ~12GB for k-means distance
# matrix → OOM. batch=16 keeps forward pass + gradients within 15GB T4.
# All k-means runs on CPU (chunked) → no GPU OOM during quantization.
# Estimated runtime: ~4 hours total

train_loader, val_loader, test_loader, class_weights = get_loaders(
    df_meta, IMAGE_PATHS, batch_size=16, num_workers=2)

result, model, quantizer, pruner = run_pipeline(
    model_fn       = build_model,
    model_name     = MODEL_NAME,
    train_loader   = train_loader,
    val_loader     = val_loader,
    test_loader    = test_loader,
    class_weights  = class_weights,
    finetune_epochs = 25,
    retrain_epochs  = 10,
    quant_ft_epochs = 10,
)

import json
with open(f"{OUT}/{MODEL_NAME}_results.json", "w") as f:
    json.dump({k:v for k,v in result.items() if k != "hist_ft"}, f, indent=2)
print(f"\n✓ Saved {OUT}/{MODEL_NAME}_results.json")


Split → Train:6,981  Val:1,532  Test:1,502

  MODEL: VGG16
  Original size: 524567.3 KB  (512.3 MB)

[1/6] Fine-tuning (25 epochs)
  Single-phase, differential LR: features=1e-3, classifier=1e-2
  [VGG16] Ep  1/25  loss=0.9978  train=36.0%  balacc=41.00%  LR(feat=0.00100 cls=0.0100)
    ★akiec=74%  ★bcc=56%   bkl=0%   df=67%  ★mel=18%   nv=0%   vasc=73%
  [VGG16] Ep  2/25  loss=0.6533  train=48.1%  balacc=46.04%  LR(feat=0.00100 cls=0.0100)
    ★akiec=58%  ★bcc=52%   bkl=11%   df=75%  ★mel=44%   nv=0%   vasc=82%
  [VGG16] Ep  3/25  loss=0.4946  train=54.5%  balacc=43.91%  LR(feat=0.00100 cls=0.0100)
    ★akiec=92%  ★bcc=41%   bkl=3%   df=38%  ★mel=60%   nv=0%   vasc=73%
  [VGG16] Ep  4/25  loss=0.4164  train=57.6%  balacc=48.71%  LR(feat=0.00100 cls=0.0100)
    ★akiec=79%  ★bcc=71%   bkl=22%   df=46%  ★mel=51%   nv=4%   vasc=68%
  [VGG16] Ep  5/25  loss=0.3562  train=61.8%  balacc=56.14%  LR(feat=0.00100 cls=0.0100)
    ★akiec=53%  ★bcc=87%   bkl=28%   df=46%  ★mel=79%   nv=19%   vasc=

In [25]:
# ── Cell 13: Figures ──────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

STAGE_KEYS   = ["base","pruned","quant","final"]
STAGE_LABELS = ["Baseline","After Pruning","After Quant.","After Huffman"]
COLORS       = ["#4C72B0","#DD8452","#55A868","#C44E52"]
plt.style.use("seaborn-v0_8-whitegrid")

# Fig 1: Per-class sensitivity all stages
fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle(f"Per-Class Sensitivity at Each Compression Stage — {MODEL_NAME}\n"
             "★ red labels = HIGH clinical risk  |  Dashed = 80% threshold",
             fontsize=13, fontweight="bold")
x = np.arange(N_CLASSES); w = 0.2
for si, (sk, sl) in enumerate(zip(STAGE_KEYS, STAGE_LABELS)):
    vals = [result[f"{sk}_sens"].get(c,0) for c in CLASS_NAMES]
    ax.bar(x+(si-1.5)*w, vals, w, label=sl,
           color=COLORS[si], alpha=0.85, edgecolor="black", lw=0.5)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=11)
for tick, c in zip(ax.get_xticklabels(), CLASS_NAMES):
    if RISK[c]=="HIGH": tick.set_color("darkred"); tick.set_fontweight("bold")
ax.axhline(80, color="red", ls="--", lw=1.5, alpha=0.7, label="80% threshold")
ax.set_ylim(0, 115); ax.set_ylabel("Sensitivity / Recall (%)", fontsize=11)
ax.legend(fontsize=9, loc="lower right")
plt.tight_layout()
plt.savefig(f"{OUT}/{MODEL_NAME}_fig1_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show(); print(f"✓ {MODEL_NAME}_fig1_sensitivity.png")

# Fig 2: Sensitivity drop heatmap
fig, ax = plt.subplots(figsize=(10, 3))
fig.suptitle(f"Sensitivity Δ% vs Baseline — {MODEL_NAME}\n"
             "Negative = Clinically Dangerous  |  Bold red = HIGH RISK",
             fontsize=11, fontweight="bold")
data = [[round(result[f"{sk}_sens"].get(c,0)-result["base_sens"].get(c,0),1)
         for c in CLASS_NAMES] for sk in STAGE_KEYS[1:]]
df_h = pd.DataFrame(data,
                    index=["After Pruning","After Quant.","After Huffman"],
                    columns=CLASS_NAMES)
sns.heatmap(df_h, ax=ax, cmap="RdYlGn", center=0,
            annot=True, fmt=".1f", cbar_kws={"label":"Δ%"}, linewidths=0.5)
for tick, c in zip(ax.get_xticklabels(), CLASS_NAMES):
    if RISK[c]=="HIGH": tick.set_color("darkred"); tick.set_fontweight("bold")
plt.tight_layout()
plt.savefig(f"{OUT}/{MODEL_NAME}_fig2_sensitivity_drop.png", dpi=150, bbox_inches="tight")
plt.show(); print(f"✓ {MODEL_NAME}_fig2_sensitivity_drop.png")

# Fig 3: Compression vs balanced accuracy + melanoma sensitivity
fig, ax = plt.subplots(figsize=(10, 5))
sizes  = [result["orig_kb"],result["pruned_kb"],result["quant_kb"],result["huff_kb"]]
ratios = [result["orig_kb"]/s for s in sizes]
bals   = [result[f"{k}_bal"] for k in STAGE_KEYS]
mels   = [result[f"{k}_sens"].get("mel",0) for k in STAGE_KEYS]
ax2    = ax.twinx()
ax.plot(ratios, bals, "o-", color="#4C72B0", lw=2.5, ms=10, label="Balanced Accuracy")
ax2.plot(ratios, mels,"s--",color="#C44E52", lw=2.5, ms=10, label="Melanoma Sensitivity")
for r,b,sl in zip(ratios,bals,STAGE_LABELS):
    ax.annotate(sl,(r,b),textcoords="offset points",xytext=(4,6),fontsize=9)
ax2.axhline(80, color="#C44E52", ls=":", lw=1.2, alpha=0.6, label="80% threshold")
ax.set_xlabel("Compression Ratio (×)", fontsize=11)
ax.set_ylabel("Balanced Accuracy (%)", color="#4C72B0", fontsize=11)
ax2.set_ylabel("Melanoma Sensitivity (%)", color="#C44E52", fontsize=11)
ax.set_title(f"Compression-Accuracy Trade-off — {MODEL_NAME}", fontsize=12, fontweight="bold")
lines = ax.get_lines()+ax2.get_lines()
ax.legend(lines,[l.get_label() for l in lines],fontsize=9,loc="center right")
plt.tight_layout()
plt.savefig(f"{OUT}/{MODEL_NAME}_fig3_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show(); print(f"✓ {MODEL_NAME}_fig3_tradeoff.png")

# Fig 4: Pipeline waterfall
fig, ax = plt.subplots(figsize=(10, 5))
sizes = [result["orig_kb"],result["pruned_kb"],result["quant_kb"],result["huff_kb"]]
WCOLS = ["#90CAF9","#FFB74D","#A5D6A7","#EF9A9A"]
bars  = ax.bar(STAGE_LABELS, sizes, color=WCOLS, edgecolor="black", lw=0.8)
for bar,sz in zip(bars,sizes):
    ax.text(bar.get_x()+bar.get_width()/2, sz*1.08,
            f"{sz:.0f} KB", ha="center", fontsize=10, fontweight="bold")
ax.set_yscale("log"); ax.set_ylabel("Size (KB, log scale)", fontsize=11)
ax.set_title(f"Compression Pipeline — {MODEL_NAME}  →  "
             f"{result['huff_ratio']:.1f}× compression",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT}/{MODEL_NAME}_fig4_waterfall.png", dpi=150, bbox_inches="tight")
plt.show(); print(f"✓ {MODEL_NAME}_fig4_waterfall.png")

print(f"\n{'='*60}")
print(f"  All figures saved to {OUT}/")
print(f"{'='*60}")


✓ VGG16_fig1_sensitivity.png
✓ VGG16_fig2_sensitivity_drop.png
✓ VGG16_fig3_tradeoff.png
✓ VGG16_fig4_waterfall.png

  All figures saved to /kaggle/working/outputs/
